# HR-1 頭打ちの確認（DP4・DP5・DP7・DP9 の記録の見直し）

> ## 🔑 先に許可しておくシークレット
> - **`GITHUB_TOKEN`（必須）**：リポジトリを決まったコミットで取ってきます。
>
> 左の鍵のマークで「ノートブックからのアクセス」をオンにしてから「すべてのセルを実行」してね。

**事後の分析**です（計画は計算の前に凍結：`notes/hr1_headroom_plan.md`、2026-10-18 16:36 JST）。新しく生成はしません。**GPU も Claude の API も使いません（費用 $0）**。

- Drive に残っている DP4・DP5・DP7・DP9 の記録を、それぞれ**回したときと同じコード**で読み直します。各実験の結果をもう一度計算し、記録した結果（`reports/dpN_RESULT.json`）と合ったときだけ使います。
- 元の「top e に対するエントロピーの AUROC」を計算し直し、記録の値と合わなければ止まります。
- 合ったら、「変わる余地（1 − q_keep）」をそろえても逆向きが残るかを見ます（H1〜H4）。
- 結果は `DECISION_POINTS/hr1/HR1_RESULT.json` に**数だけ**書きます。ほかのフォルダは書き換えません。問題の中身は表示しません。

**使い方**：CPU のランタイム（ハイメモリは不要）。「すべてのセルを実行」で 15〜30 分（Drive の記録を読むのに時間がかかります）。終わったら Claude に教えてね。

- **通知**：スマホの ntfy アプリで `godpork-b06080cfb3cdfa8f` を購読しておくと、始まったとき・終わったとき・エラーで止まったときに届きます。ランタイムごと切れたときは届かないので、そのときは Colab を見てください。
- **終わったら自動でランタイムを切ります**（CU の節約）。Drive への保存を確定してから切るので、結果は残ります。エラーで止まったときは切らずに残します。切りたくないときは、最初に `AUTO_DISCONNECT = False` を実行してください。
- **エラーのとき**：通知の本文にエラーの文面（トレースバック）が入ります。全文は Drive の `DECISION_POINTS/_errors/` にも保存されるので、Claude が直接読めます。
- 通知の頭には 🚀／⚠／🎉 が付きます。

In [ ]:
# Cell 1 — Drive, clone this head, one worktree per experiment at its own run commit, re-read, check, analyse.
from google.colab import drive, userdata
from pathlib import Path
import os, shutil, stat, subprocess, sys, json, time, hashlib, datetime as dt

# 通知（ntfy）：スマホの ntfy アプリでこの名前を購読しておくと、始まったとき・終わったとき・止まったときに届きます。エラーのときはエラーの文面も送ります（鍵・トークンらしい文字は伏せ字。全文は Drive の DECISION_POINTS/_errors）
import urllib.request
NTFY_TOPIC = 'godpork-b06080cfb3cdfa8f'
NB_LABEL = 'HR-1 頭打ちの確認（DP4・DP5・DP7・DP9 の記録、CPU）'   # どのノートブックからの通知かの名札
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=('[' + NB_LABEL + '] ' + msg).encode('utf-8'),
                                     headers={'Title': 'GodPork - Pork HR-1 headroom', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
import os, re as _re, traceback as _tb, datetime as _edt
ERR_SLUG = 'HR1_HEADROOM'   # Drive のエラーファイルの名前の頭
_SECRET_RE = _re.compile(r"sk-ant-[A-Za-z0-9_\-]+|github_pat_[A-Za-z0-9_]+|gh[pousr]_[A-Za-z0-9]{8,}|hf_[A-Za-z0-9]{8,}|x-access-token:[^@\s]+@")
def _scrub(s):
    s = _SECRET_RE.sub('[伏せ字]', s)
    return '\n'.join(l if len(l) <= 300 else l[:300] + ' …（長いので省略）' for l in s.splitlines())
def _error_text(result):
    e = result.error_in_exec if result.error_in_exec is not None else result.error_before_exec
    tb = ''.join(_tb.format_exception(type(e), e, e.__traceback__))
    first = (getattr(getattr(result, 'info', None), 'raw_cell', '') or '').strip().splitlines()[:1]
    return _scrub('セル: ' + (first[0][:100] if first else '?') + '\n' + tb)
def _save_error(text):
    base = '/content/drive/MyDrive'
    if not os.path.isdir(base):
        return None
    d = os.path.join(base, 'DECISION_POINTS', '_errors')
    os.makedirs(d, exist_ok=True)
    stem = ERR_SLUG + '_' + _edt.datetime.now(_edt.timezone(_edt.timedelta(hours=9))).strftime('%Y%m%d_%H%M%S')
    name, n = stem + '.txt', 1
    while os.path.exists(os.path.join(d, name)):
        n += 1
        name = stem + '_' + str(n) + '.txt'
    with open(os.path.join(d, name), 'w', encoding='utf-8') as fh:
        fh.write('[' + NB_LABEL + ']\n' + text)
    return 'DECISION_POINTS/_errors/' + name
def _short(text, limit=3300):
    b = text.encode('utf-8')
    if len(b) <= limit:
        return text
    return '…（前のほうは省略。全文は Drive）\n' + b[-limit:].decode('utf-8', 'ignore')
def _on_error(result):
    if result.error_in_exec is None and result.error_before_exec is None:
        return
    try:
        text = _error_text(result)
    except Exception as e2:
        text = 'エラーの文面を作れませんでした：' + type(e2).__name__
    try:
        where = _save_error(text)
    except Exception as e3:
        where = None
        text += '\n（Drive に保存できませんでした：' + type(e3).__name__ + '）'
    print('エラーの全文を通知に送ります' + ('（Drive：' + where + '）' if where else ''))
    notify('⚠ 止まりました：エラーが出ています。' + ('全文は Drive の ' + where if where else 'Drive には保存できませんでした')
           + '\n\n' + _short(text), tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

drive.mount("/content/drive")

EXPECTED_HEAD = "c1542edcc75419cca9ecc02b2b1dd07b7b35bd6e"
EXPS = json.loads(r"""{"dp4": ["dp4_weighting", "dp4_runner", ["A", "B"], "dp4/DP4_1B_{h}_20261008", "b64a0065590662c2270d110c80ff095f41f80f5e", []], "dp5": ["dp5_gemma", "dp5_runner", ["A", "B", "C"], "dp5/DP5_GEMMA_{h}_20261008", "fd5b17566df21a1c0522afa53b97f4af8b42d552", []], "dp7": ["dp7_matched", "dp7_runner", ["A", "B"], "dp7/DP7_GEMMA_{h}_20261010", "5592aa76bdd1d0576af563f241f2e31c96d33511", ["matched"]], "dp9": ["dp9_qwen", "dp9_runner", ["A", "B", "C", "D"], "dp9/DP9_QWEN_{h}_20261017", "43ef025fdedc964048a596536ce180139e4f549f", []]}""")
COLS = json.loads(r"""["row", "template", "t", "entropy", "q_keep", "q_swap"]""")
BRANCH = "decision-points"
REPO_URL = "https://github.com/chichichi2001/Pork.git"
REPO = Path("/content/Pork")
THEME = Path("/content/drive/MyDrive/DECISION_POINTS")
OUT = THEME / "hr1"
WORK = Path("/content/hr1_work")
for name, (_m, _r, parts, fmt, _c, _x) in EXPS.items():
    for h in parts:
        assert (THEME / fmt.format(h=h) / "RUN_IDENTITY.json").exists(), f"{name} の実行フォルダ {h} が見つかりません"

try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
assert token and str(token).strip(), "Colab のシークレットに GITHUB_TOKEN を登録してね（ノートブックへのアクセスもオン）。"
askpass = Path("/tmp/pork_git_askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *Password*) echo "$GITHUB_TOKEN" ;;\n  *) echo "" ;;\nesac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
env = dict(os.environ, GITHUB_TOKEN=str(token).strip(), GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
token = None
os.chdir('/content')   # セル1 をやり直しても、消すフォルダの中にいないように（2026-10-14）
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "-q", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO)], check=True, env=env)
env.pop("GITHUB_TOKEN", None)
subprocess.run(["git", "-C", str(REPO), "checkout", "-q", "--detach", EXPECTED_HEAD], check=True)
head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert head == EXPECTED_HEAD, (head, EXPECTED_HEAD)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyarrow", "scipy"], capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-3000:]
sys.path.insert(0, str(REPO))
from pork import hr1_headroom as H
assert H.plan_doc_sha256(REPO) == H.PLAN_DOC_SHA256, "HR-1 の計画が凍結したものと違います"
print("head", head[:12])

if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)
(WORK / "extract.py").write_text(r"""import importlib, json, sys
from pathlib import Path
import numpy as np
spec = json.loads(sys.argv[1])
d = importlib.import_module("pork." + spec["module"]); rr = importlib.import_module("pork." + spec["runner"])
if spec["name"] == "dp9":
    from pork import dp2_real_forks as d2
    rows_raw, _ = d2.load_rows_from_hub()
    items = d.select_items(rows_raw); del rows_raw
    split = d.parts(); part_key = "part"
else:
    rows_raw, _ = d.load_rows_from_hub()
    items = d.select_items(rows_raw); del rows_raw
    split = d.halves(); part_key = "half"
items_sha = d.items_sha256(items)
cases, commits = [], {}
for h in spec["parts"]:
    b = Path(spec["base_fmt"].format(h=h))
    ident = json.loads((b / "RUN_IDENTITY.json").read_text())
    assert ident["items_sha256"] == items_sha and ident[part_key] == h and ident["gate_doc_sha256"] == d.GATE_DOC_SHA256, h
    assert not any(b.glob("RUN_FAILED*.txt")), h
    cases += rr.load_cases(split[h], units_dir=b / "units", code_commit=ident["code_commit"], items_hash=items_sha)
    commits[h] = ident["code_commit"]
cases = sorted(cases, key=lambda c: c["row"])
assert [c["row"] for c in cases] == sorted(r for h in spec["parts"] for r in split[h])
analysis = json.loads(json.dumps(d.analyze(cases), sort_keys=True))
tab = d.position_table(cases)
out = Path(spec["out"]); out.mkdir(parents=True, exist_ok=True)
np.savez(out / "table.npz", **{k: np.asarray(tab[k]) for k in spec["cols"]})
(out / "analysis.json").write_text(json.dumps(analysis, sort_keys=True))
(out / "identity.json").write_text(json.dumps({"items_sha256": items_sha, "code_commits": commits,
                                               "gate_doc_sha256": d.GATE_DOC_SHA256, "schema": d.SCHEMA}, sort_keys=True))
print("extracted", spec["name"], "positions", len(tab["row"]))
""")
TABLES, CHECKS = {}, {}
for name, (module, runner, parts, fmt, commit, extra) in EXPS.items():
    wt = WORK / ("wt_" + name)
    subprocess.run(["git", "-C", str(REPO), "worktree", "add", "-q", "--detach", str(wt), commit], check=True)
    spec = {"name": name, "module": module, "runner": runner, "parts": parts, "base_fmt": str(THEME / fmt),
            "out": str(WORK / ("out_" + name)), "cols": COLS + extra}
    p = subprocess.run([sys.executable, str(WORK / "extract.py"), json.dumps(spec)], cwd=str(wt),
                       env=dict(os.environ, PYTHONPATH=str(wt)), capture_output=True, text=True)
    assert p.returncode == 0, name + " の読み直しに失敗しました:\n" + p.stderr[-3000:]
    print(p.stdout.strip().splitlines()[-1])
    o = WORK / ("out_" + name)
    analysis = json.loads((o / "analysis.json").read_text()); ident = json.loads((o / "identity.json").read_text())
    stored = json.loads((REPO / "reports" / (name + "_RESULT.json")).read_text())
    st = {k: v for k, v in stored.items() if k != "result_sha256"}
    assert hashlib.sha256(json.dumps(st, sort_keys=True).encode()).hexdigest() == stored["result_sha256"], name + " stored"
    assert set(ident["code_commits"].values()) == {commit}, name
    assert ident["items_sha256"] == stored["items_sha256"] and ident["gate_doc_sha256"] == stored["gate_doc_sha256"], name
    rebuilt = {"schema": ident["schema"], "analysis": analysis, "items_sha256": ident["items_sha256"],
               "gate_doc_sha256": ident["gate_doc_sha256"], "versions": stored["versions"]}
    if "code_commit" in stored:                     # DP9 records one commit
        assert stored["code_commit"] == commit, name
        rebuilt["code_commit"] = commit
    else:
        assert ident["code_commits"] == stored["code_commits"], name
        rebuilt["code_commits"] = ident["code_commits"]
    exact = hashlib.sha256(json.dumps(rebuilt, sort_keys=True).encode()).hexdigest() == stored["result_sha256"]
    from pork import dp8x_explore as X
    diffs = [] if exact else X.same_numbers(analysis, stored["analysis"], tol=1e-9)
    assert exact or not diffs, f"{name}: 記録した結果と合いません（{len(diffs)} か所）— Claude に見せてね"
    CHECKS[name] = {"result_sha256_exact": exact, "within_1e-9": True, "stored_result_sha256": stored["result_sha256"]}
    print(name, "記録と照合:", "完全に一致" if exact else "丸めの違いだけ（1e-9 以内）")
    z = __import__("numpy").load(o / "table.npz")
    TABLES[name] = {k: z[k] for k in z.files}
    TABLES[name + "__stored"] = stored["analysis"]

stored_analyses = {k[:-8]: TABLES.pop(k) for k in list(TABLES) if k.endswith("__stored")}
r1 = H.analyze_all(TABLES); r2 = H.analyze_all(TABLES)
assert H.result_sha(r1) == H.result_sha(r2), "集計が再現しません"
ORIG = {name: H.check_original(name, r1["experiments"][name], stored_analyses[name]) for name in TABLES}
bad = [n for n, c in ORIG.items() if not c["match"]]
assert not bad, "元の AUROC を計算し直した値が記録と合いません（止める線）: " + ", ".join(bad) + " — Claude に見せてね"
import numpy as np, scipy
res = {"hr1": r1, "reread_checks": CHECKS, "original_checks": ORIG, "code_commit": EXPECTED_HEAD,
       "plan_doc_sha256": H.PLAN_DOC_SHA256, "versions": {"numpy": np.__version__, "scipy": scipy.__version__}}
res["result_sha256"] = H.result_sha(res)
OUT.mkdir(parents=True, exist_ok=True)
tmp = OUT / "HR1_RESULT.json.tmp"
tmp.write_text(json.dumps(res, sort_keys=True, indent=1))
os.replace(tmp, OUT / "HR1_RESULT.json")
f = lambda v: "  —  " if v is None else f"{v:.3f}"
ci = lambda c: "[" + f(c[0]) + ", " + f(c[1]) + "]"
READ = {"REMAINS": "残る", "CANNOT_EXCLUDE": "否定できない", "REVERSES": "逆になる", "NO_INTERVAL": "区間なし"}
for name, e in r1["experiments"].items():
    o, h2, h3, h1 = e["original"], e["H2_headroom_normalised"], e["H3_ample_headroom"], e["H1_spearman_entropy_q_keep"]
    print(f"\n[{name}] 位置 {e['positions']}・問題 {e['items']}")
    print(f"  元の top e      : {f(o['within_item_auroc'])} {ci(o['ci'])}（記録と一致）")
    print(f"  H2 余地で割る   : {f(h2['within_item_auroc'])} {ci(h2['ci'])}（除いた位置 {h2['excluded_no_headroom']}）")
    print(f"  H3 余地が十分   : {f(h3['within_item_auroc'])} {ci(h3['ci'])}（残した位置 {h3['positions']}）")
    print(f"  H1 エントロピーと q_keep のスピアマン: {f(h1['mean'])} {ci(h1['ci'])}")
    print("  読み方:", READ[e["reading"]])
print("\n結果 sha256:", res["result_sha256"][:16], "| 保存: DECISION_POINTS/hr1/HR1_RESULT.json")
print("✅ 終わり。Claude に知らせてね")
notify('🎉 終わりました：HR-1 頭打ちの確認（' + '・'.join(k + ' ' + READ[e['reading']] for k, e in r1['experiments'].items()) + '）', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    print('Drive への書き込みを確定してから、ランタイムを切ります（CU の節約。止めたいときは AUTO_DISCONNECT = False）')
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount()
    time.sleep(3)
    _runtime.unassign()